# Stage 1 — Data Exploration & Entity Mapping

**Goals:**
1. Download IBM AML transactions / patterns and inspect their structure
2. Download ICIJ Offshore Leaks and verify jurisdiction distribution
3. Extract three demo attempts (CYCLE / FAN-IN / FAN-OUT)
4. Decide IBM account ↔ ICIJ jurisdiction mapping
5. Persist intermediate artifacts (`stage1_data.pkl/json` → mirrored to Drive)

**Next stage:** Stage 2 — Raw Store construction (entity / account / transaction / ip / registration RawRecords)

## 0. Kaggle Authentication Setup

In [ ]:
# Upload the Kaggle API token (kaggle.json) — run once per session
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle

## 1. Download IBM AML Dataset

In [ ]:
!kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml
!unzip -q ibm-transactions-for-anti-money-laundering-aml.zip -d aml_data/
!ls -lh aml_data/

## 2. Inspect Transactions CSV Schema

We use `HI-Small_Trans.csv` for the demo (smallest variant, high illicit-ratio). The actual column names use spaces (`Is Laundering`, `From Bank`) rather than the underscores assumed in the v6 design document — note the divergence.

In [ ]:
import pandas as pd

df = pd.read_csv('aml_data/HI-Small_Trans.csv', nrows=1000)
print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nDtypes:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

## 3. Label Distribution + Currency / Time / Account Stats

In [ ]:
df_full = pd.read_csv('aml_data/HI-Small_Trans.csv')
print('Total transactions:', len(df_full))
print(f"Memory: {df_full.memory_usage(deep=True).sum()/1e6:.1f} MB")

print('\n=== Is Laundering distribution ===')
print(df_full['Is Laundering'].value_counts())
print(f"Laundering ratio: {df_full['Is Laundering'].mean()*100:.3f}%")

print('\n=== Currency pairs (top 10) ===')
print(df_full.groupby(['Payment Currency', 'Receiving Currency']).size().sort_values(ascending=False).head(10))

print('\n=== Payment Format distribution ===')
print(df_full['Payment Format'].value_counts())

print('\n=== Time range ===')
print('Min:', df_full['Timestamp'].min())
print('Max:', df_full['Timestamp'].max())

print(f"\nUnique 'From Bank': {df_full['From Bank'].nunique()}")
print(f"Unique 'To Bank': {df_full['To Bank'].nunique()}")
print(f"Unique 'Account' (sender): {df_full['Account'].nunique()}")
print(f"Unique 'Account.1' (receiver): {df_full['Account.1'].nunique()}")

## 4. Inspect Patterns.txt Format

**Key finding:** the file is structured as `BEGIN LAUNDERING ATTEMPT - <PATTERN_TYPE>` ~ `END LAUNDERING ATTEMPT - <PATTERN_TYPE>` blocks, with transaction lines listed in the same format as transactions.csv. This is **explicit ground-truth labeling**, which means our Pattern Detector can consume these labels directly without independent detection logic.

In [ ]:
with open('aml_data/HI-Small_Patterns.txt', 'r') as f:
    for i, line in enumerate(f):
        if i >= 100: break
        print(line.rstrip())

import os
print(f"\nFile size: {os.path.getsize('aml_data/HI-Small_Patterns.txt')/1024:.1f} KB")

## 5. Pattern Type Distribution

In [ ]:
import re
from collections import defaultdict, Counter

pattern_types = Counter()
pattern_tx_counts = defaultdict(list)

current_type = None
current_count = 0

with open('aml_data/HI-Small_Patterns.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('BEGIN LAUNDERING ATTEMPT'):
            m = re.match(r'BEGIN LAUNDERING ATTEMPT\s*-\s*([A-Z\-]+)', line)
            if m:
                current_type = m.group(1)
                current_count = 0
        elif line.startswith('END LAUNDERING ATTEMPT'):
            if current_type:
                pattern_types[current_type] += 1
                pattern_tx_counts[current_type].append(current_count)
            current_type = None
            current_count = 0
        elif current_type and line:
            current_count += 1

print('=== Pattern type distribution (number of attempts) ===')
for ptype, cnt in pattern_types.most_common():
    avg_tx = sum(pattern_tx_counts[ptype]) / len(pattern_tx_counts[ptype])
    print(f'  {ptype:25s}  {cnt:>6,} attempts  (avg {avg_tx:.1f} tx per attempt)')

print(f'\nTotal laundering attempts: {sum(pattern_types.values()):,}')
print(f'Total laundering transactions: {sum(sum(c) for c in pattern_tx_counts.values()):,}')

## 6. Extract CYCLE Attempt Candidates (4–6 hops)

We pick a moderate-sized cycle for the demo. Too many cards on the board crowd the visualization.

In [ ]:
cycle_attempts = []

with open('aml_data/HI-Small_Patterns.txt', 'r') as f:
    in_cycle = False
    current_lines = []
    for line in f:
        line = line.rstrip()
        if line.startswith('BEGIN LAUNDERING ATTEMPT - CYCLE'):
            in_cycle = True
            current_lines = [line]
            continue
        if in_cycle:
            current_lines.append(line)
            if line.startswith('END LAUNDERING ATTEMPT - CYCLE'):
                tx_count = len(current_lines) - 2
                cycle_attempts.append({'tx_count': tx_count, 'lines': current_lines})
                in_cycle = False

hop_counts = [a['tx_count'] for a in cycle_attempts]
print('=== CYCLE hop count distribution ===')
for hop, cnt in sorted(Counter(hop_counts).items()):
    print(f'  {hop} hops: {cnt} attempts')

demo_candidates = [a for a in cycle_attempts if 4 <= a['tx_count'] <= 6]
print(f'\n=== Demo candidates (4-6 hops): {len(demo_candidates)} attempts ===')

for i, c in enumerate(demo_candidates[:3]):
    print(f"\n--- Candidate {i+1} ({c['tx_count']} hops) ---")
    for line in c['lines']:
        print(line)

## 7. FAN-IN / FAN-OUT Candidate Preview

In [ ]:
def collect_attempts(filepath, pattern_name):
    attempts = []
    with open(filepath, 'r') as f:
        in_block = False
        current_lines = []
        for line in f:
            line = line.rstrip()
            if line.startswith(f'BEGIN LAUNDERING ATTEMPT - {pattern_name}'):
                in_block = True
                current_lines = [line]
                continue
            if in_block:
                current_lines.append(line)
                if line.startswith(f'END LAUNDERING ATTEMPT - {pattern_name}'):
                    attempts.append({'tx_count': len(current_lines) - 2, 'lines': current_lines})
                    in_block = False
    return attempts

for pname in ['FAN-IN', 'FAN-OUT']:
    attempts = collect_attempts('aml_data/HI-Small_Patterns.txt', pname)
    sized = [a for a in attempts if 3 <= a['tx_count'] <= 5]
    print(f'\n=== {pname}: {len(attempts)} total, {len(sized)} with 3-5 tx ===')
    if sized:
        c = sized[0]
        print(f"--- Sample ({c['tx_count']} tx) ---")
        for line in c['lines']:
            print(line)

## 8. Confirm and Structure the Three Demo Attempts

**Final selection:**
- **CYCLE (main)** = Candidate 1 (6-hop, Euro / Yen / Brazil Real)
- **FAN-IN (auxiliary)** = first 3-tx attempt
- **FAN-OUT (auxiliary)** = first 4-tx attempt

In [ ]:
import re

def parse_attempt_block(lines):
    header = lines[0]
    m = re.match(r'BEGIN LAUNDERING ATTEMPT\s*-\s*([A-Z\-]+):\s*(.*)', header)
    pattern_type = m.group(1) if m else 'UNKNOWN'
    description = m.group(2) if m else ''

    transactions = []
    for line in lines[1:-1]:
        parts = line.split(',')
        if len(parts) < 11:
            continue
        transactions.append({
            'timestamp': parts[0],
            'from_bank': parts[1],
            'from_account': parts[2],
            'to_bank': parts[3],
            'to_account': parts[4],
            'amount_received': float(parts[5]),
            'receiving_currency': parts[6],
            'amount_paid': float(parts[7]),
            'payment_currency': parts[8],
            'payment_format': parts[9],
            'is_laundering': int(parts[10]),
        })

    accounts = set()
    for tx in transactions:
        accounts.add((tx['from_bank'], tx['from_account']))
        accounts.add((tx['to_bank'], tx['to_account']))

    return {
        'pattern_type': pattern_type,
        'description': description,
        'transactions': transactions,
        'unique_accounts': sorted(accounts),
        'tx_count': len(transactions),
    }

cycle_demo = parse_attempt_block(demo_candidates[0]['lines'])
fan_in_attempts = collect_attempts('aml_data/HI-Small_Patterns.txt', 'FAN-IN')
fan_out_attempts = collect_attempts('aml_data/HI-Small_Patterns.txt', 'FAN-OUT')
fanin_demo = parse_attempt_block([a for a in fan_in_attempts if a['tx_count'] == 3][0]['lines'])
fanout_demo = parse_attempt_block([a for a in fan_out_attempts if a['tx_count'] == 4][0]['lines'])

for name, demo in [('CYCLE', cycle_demo), ('FAN-IN', fanin_demo), ('FAN-OUT', fanout_demo)]:
    print(f"=== {name} ({demo['pattern_type']}, {demo['tx_count']} tx) ===")
    print(f"Description: {demo['description']}")
    print(f"Unique accounts ({len(demo['unique_accounts'])}):")
    for bank, acct in demo['unique_accounts']:
        print(f'  bank={bank}, account={acct}')
    print()

all_accounts = set()
for demo in [cycle_demo, fanin_demo, fanout_demo]:
    all_accounts.update(demo['unique_accounts'])
print(f'=== Total unique accounts across 3 patterns: {len(all_accounts)} ===')

## 9. Currency Distribution (jurisdiction-mapping hint)

In [ ]:
from collections import Counter

for name, demo in [('CYCLE', cycle_demo), ('FAN-IN', fanin_demo), ('FAN-OUT', fanout_demo)]:
    currencies = [tx['receiving_currency'] for tx in demo['transactions']]
    print(f'{name}: {dict(Counter(currencies))}')

## 10. Download ICIJ Offshore Leaks

In [ ]:
!wget -q https://offshoreleaks-data.icij.org/offshoreleaks/csv/full-oldb.LATEST.zip
!ls -lh full-oldb.LATEST.zip
!unzip -q full-oldb.LATEST.zip -d icij_data/
!ls -lh icij_data/

## 11. ICIJ Entities Schema + Jurisdiction Distribution

In [ ]:
ent_full = pd.read_csv('icij_data/nodes-entities.csv',
                       usecols=['node_id', 'name', 'jurisdiction',
                                'jurisdiction_description', 'service_provider',
                                'address', 'sourceID', 'incorporation_date'],
                       low_memory=False)

print(f'Total entities: {len(ent_full):,}\n')
print('=== jurisdiction_description top 30 ===')
print(ent_full['jurisdiction_description'].value_counts().head(30))

## 12. Sample Entities from 8 Demo Jurisdictions

Intersection between scenario-relevant jurisdictions and ICIJ-rich jurisdictions:

In [ ]:
DEMO_JURISDICTIONS = {
    'BVI':       'British Virgin Islands',
    'Bahamas':   'Bahamas',
    'Malta':     'Malta',
    'Panama':    'Panama',
    'Cayman':    'Cayman Islands',
    'Singapore': 'Singapore',
    'Delaware':  'State of Delaware',
    'Cyprus':    'Cyprus',
}

def sample_entities(df, juris_desc, n, seed=42):
    pool = df[df['jurisdiction_description'] == juris_desc].copy()
    if len(pool) > n:
        pool = pool.sample(n=n, random_state=seed)
    return pool

samples = {}
for short, full in DEMO_JURISDICTIONS.items():
    samp = sample_entities(ent_full, full, n=10)
    samples[short] = samp
    print(f"=== {short} ({full}) — pool: {(ent_full['jurisdiction_description']==full).sum():,}, sampled: {len(samp)} ===")
    if len(samp) > 0:
        print(samp[['node_id', 'name', 'service_provider', 'sourceID']].head(3).to_string())
    print()

## 13. Decide IBM Account → ICIJ Jurisdiction Mapping

We match transaction currency flow to plausible shell-company jurisdictions. For example, accounts receiving Yen map to BVI / Cayman (asset-management SPCs frequently use these); accounts receiving Brazil Real map to Panama (a common conduit for South American capital).

In [ ]:
print('=== CYCLE flow ===')
for i, tx in enumerate(cycle_demo['transactions']):
    print(f"  hop {i+1}: {tx['from_account']:12s} \u2192 {tx['to_account']:12s} "
          f"{tx['receiving_currency']:15s} {tx['amount_received']:>14,.2f}")

ACCOUNT_TO_JURISDICTION = {
    # CYCLE
    '803A568D0': 'Malta',     # cycle start / end (closure point)
    '8012F9500': 'Bahamas',
    '8102CA2C0': 'Cyprus',
    '8040FAF80': 'BVI',       # receives Yen
    '80AC28E60': 'BVI',
    '80CF37D80': 'Panama',    # receives Brazil Real
    # FAN-IN — hub at BVI
    '811C597B0': 'BVI',       # hub (incoming only)
    '811C599A0': 'Cayman',
    '811B83280': 'Bahamas',
    '812D22980': 'Singapore',
    # FAN-OUT — hub at Delaware
    '80769F1A0': 'Delaware',  # hub (outgoing only)
    '804529040': 'BVI',
    '811A9BD50': 'Singapore',
    '800BFCD30': 'BVI',
    '8020BC410': 'Cayman',
}

all_ibm_accounts = set()
for demo in [cycle_demo, fanin_demo, fanout_demo]:
    for bank, acct in demo['unique_accounts']:
        all_ibm_accounts.add(acct)

unmapped = all_ibm_accounts - set(ACCOUNT_TO_JURISDICTION.keys())
extra = set(ACCOUNT_TO_JURISDICTION.keys()) - all_ibm_accounts
print(f'\nIBM accounts total: {len(all_ibm_accounts)}')
print(f'Mapped: {len(ACCOUNT_TO_JURISDICTION)}')
print(f"Unmapped: {unmapped if unmapped else 'none ✅'}")
print(f"Extra: {extra if extra else 'none ✅'}")

## 14. Persist Stage 1 Artifacts

Two formats: `pickle` for full Python-object roundtripping, `json` for human readability and git review.
Mirrored to Google Drive to survive Colab's 12-hour session reset.

In [ ]:
import pickle, json
from pathlib import Path

Path('artifacts').mkdir(exist_ok=True)

artifacts = {
    'cycle_demo': cycle_demo,
    'fanin_demo': fanin_demo,
    'fanout_demo': fanout_demo,
    'icij_samples': samples,
    'account_to_jurisdiction': ACCOUNT_TO_JURISDICTION,
}

with open('artifacts/stage1_data.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

artifacts_json = {
    'cycle_demo': cycle_demo,
    'fanin_demo': fanin_demo,
    'fanout_demo': fanout_demo,
    'icij_samples': {k: v.to_dict(orient='records') for k, v in samples.items()},
    'account_to_jurisdiction': ACCOUNT_TO_JURISDICTION,
}
with open('artifacts/stage1_data.json', 'w', encoding='utf-8') as f:
    json.dump(artifacts_json, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved: artifacts/stage1_data.pkl ({Path('artifacts/stage1_data.pkl').stat().st_size/1024:.1f} KB)")
print(f"Saved: artifacts/stage1_data.json ({Path('artifacts/stage1_data.json').stat().st_size/1024:.1f} KB)")

# Mirror to Drive
from google.colab import drive
drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/bakerstreet_artifacts')
drive_dir.mkdir(exist_ok=True)
import shutil
shutil.copy('artifacts/stage1_data.pkl', drive_dir)
shutil.copy('artifacts/stage1_data.json', drive_dir)
print(f'\nMirrored to: {drive_dir}')
print(f'Files: {list(drive_dir.iterdir())}')

---

## Next Stage — `02_build_raw_store.ipynb`

Using `stage1_data.pkl` as input:
1. Pick exactly one ICIJ entity per IBM account (15 entities finalized)
2. Build transaction RawRecords
3. Build account RawRecords (`holder` = ICIJ entity id)
4. Build registration RawRecords (shared service_provider, shared address)
5. Synthesize ip_log RawRecords (artificially imposed)
6. Verify — every evidence should be reverse-traceable to a record in the raw store

Output: `stage2_raw_store.json`